In [1]:
import os
import torch
import pandas as pd

# 1. Project Directory Setup
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# 2. GPU Check (Crucial for your Asus A15)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("❌ GPU NOT detected. Using CPU.")

print(f"Project folder is set to: {PROJECT_ROOT}")

❌ GPU NOT detected. Using CPU.
Project folder is set to: c:\Users\Vinindu Siriwardhana\Documents\DengueGNN


In [1]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.7.0.dev20250310+cu124
CUDA available: True
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [18]:
import torch
import numpy as np
from Bio.PDB import PDBParser
from torch_geometric.nn import radius_graph
from torch_geometric.data import Data

class PPI_ResidueEncoder:
    def __init__(self, radius=8.0):
        self.radius = radius
        self.parser = PDBParser(QUIET=True)

    def get_residue_coords(self, pdb_path):
        """Extracts the CA (Alpha Carbon) coordinates for every residue."""
        structure = self.parser.get_structure('protein', pdb_path)
        coords = []
        for model in structure:
            for chain in model:
                for residue in chain:
                    if 'CA' in residue:
                        coords.append(residue['CA'].get_coord())
        return np.array(coords)

    def build_graph(self, pdb_id, embedding_dict):
        """
        pdb_id: The ID (e.g., 'O00206') used as a key in your embeddings_cache
        embedding_dict: The cache generated by the ESM-2 script
        """
        # 1. Get pre-computed ESM-2 embeddings
        if pdb_id not in embedding_dict:
            return None
        
        x = torch.tensor(embedding_dict[pdb_id]['embedding'], dtype=torch.float)
        
        # 2. Get Coordinates from structure file
        # Assuming structures are in 'data/structures/' or 'data/dengue_structures/'
        pdb_path = f"data/structures/{pdb_id}.pdb"
        if not os.path.exists(pdb_path):
            pdb_path = f"data/dengue_structures/{pdb_id}.pdb"
            
        pos_np = self.get_residue_coords(pdb_path)
        
        # Safety check: embedding length must match coordinate length
        if len(x) != len(pos_np):
            # Sometimes AlphaFold PDBs and ESM sequences have slight mismatches
            # We truncate to the shorter length
            min_len = min(len(x), len(pos_np))
            x = x[:min_len]
            pos_np = pos_np[:min_len]

        pos = torch.tensor(pos_np, dtype=torch.float)

        # 3. Create Edges based on 8 Angstrom proximity
        edge_index = radius_graph(pos, r=self.radius, loop=False)

        return Data(x=x, edge_index=edge_index, pos=pos)

encoder = PPI_ResidueEncoder(radius=8.0)
print("PPI Residue Encoder (Residue-level) ready.")

PPI Residue Encoder (Residue-level) ready.


In [19]:
import pandas as pd
import os
import requests
import time
from tqdm import tqdm

# Create necessary directories
os.makedirs('data/structures', exist_ok=True)
os.makedirs('data/dengue_structures', exist_ok=True)

def get_string_ppis(limit=1000):
    print("Fetching human PPIs from STRING...")
    # Hub proteins to get a variety of interactions
    hubs = ['TP53', 'EGFR', 'AKT1', 'STAT3', 'TNF']
    all_ppis = []
    for hub in hubs:
        url = f"https://string-db.org/api/json/network?identifiers={hub}&species=9606&limit={limit//len(hubs)}"
        try:
            res = requests.get(url)
            if res.status_code == 200:
                for inter in res.json():
                    all_ppis.append({
                        'p1': inter.get('stringId_A', '').split('.')[1],
                        'p2': inter.get('stringId_B', '').split('.')[1],
                        'label': 1
                    })
        except: continue
    return pd.DataFrame(all_ppis).drop_duplicates()

# 1. General PPIs
pretrain_df = get_string_ppis()

# 2. Curated Dengue Binders (The "Gold" Data)
dengue_pos = pd.DataFrame([
    {'p1': 'P03314', 'p2': 'O00206', 'label': 1}, # NS1 + TLR4
    {'p1': 'P03314', 'p2': 'O60603', 'label': 1}, # NS1 + TLR2
    {'p1': 'P03314', 'p2': 'Q9NY25', 'label': 1}, # NS1 + CLEC5A
    {'p1': 'P03314', 'p2': 'P04003', 'label': 1}, # NS1 + C4BPA
])

# 3. Generate Negatives (Random pairs)
all_u = list(set(pretrain_df['p1'].tolist() + pretrain_df['p2'].tolist()))
negatives = []
import random
while len(negatives) < len(pretrain_df):
    u1, u2 = random.sample(all_u, 2)
    negatives.append({'p1': u1, 'p2': u2, 'label': 0})
neg_df = pd.DataFrame(negatives)

# Combine and save
full_dataset = pd.concat([pretrain_df, dengue_pos, neg_df])
full_dataset.to_csv('data/interaction_master_list.csv', index=False)
print(f"Saved {len(full_dataset)} interaction pairs to data/interaction_master_list.csv")

Fetching human PPIs from STRING...
Saved 60982 interaction pairs to data/interaction_master_list.csv


In [20]:
def download_pdb(uniprot_id):
    path = f"data/structures/{uniprot_id}.pdb"
    if os.path.exists(path): return True
    
    # Try AlphaFold
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            with open(path, 'wb') as f: f.write(r.content)
            return True
    except: pass
    return False

# Download NS1 specifically (4O6B)
ns1_url = "https://files.rcsb.org/download/4O6B.pdb"
with open("data/structures/P03314.pdb", "wb") as f:
    f.write(requests.get(ns1_url).content)

# Download all others in the list
u_ids = list(set(full_dataset['p1'].tolist() + full_dataset['p2'].tolist()))
print(f"Downloading {len(u_ids)} structures...")
for uid in tqdm(u_ids):
    download_pdb(uid)

100%|██████████| 739/739 [10:05<00:00,  1.22it/s]


In [28]:
import pandas as pd

df = pd.read_csv('data/interaction_master_list.csv')
# Combine p1 and p2, remove the prefix, and get unique count
all_proteins = pd.concat([df['p1'], df['p2']]).unique()
human_ids = [str(p).split('.')[-1] for p in all_proteins if '9606' in str(p) or 'ENSP' in str(p)]

print(f"Total Interactions: {len(df)}")
print(f"Unique Human Proteins involved: {len(set(human_ids))}")

Total Interactions: 60982
Unique Human Proteins involved: 734


In [29]:
import pandas as pd
import os

# 1. Load the master interaction list
df = pd.read_csv('data/interaction_master_list.csv')

# 2. Extract and Clean IDs
# We take p1 and p2, combine them, and remove the '9606.' prefix
all_involved = pd.concat([df['p1'], df['p2']]).unique()

# Filter for human proteins (ENSP) and remove prefixes
human_ensp_ids = []
for p in all_involved:
    p_str = str(p)
    if 'ENSP' in p_str:
        # Get just the ENSP000... part
        clean_id = p_str.split('.')[-1]
        human_ensp_ids.append(clean_id)

# Remove duplicates (just in case)
unique_human_ids = sorted(list(set(human_ensp_ids)))

# 3. Create a clean DataFrame
shopping_list = pd.DataFrame({
    'ensembl_protein_id': unique_human_ids,
    'uniprot_id': '', # To be filled by our mapper
    'pdb_downloaded': 'No',
    'source': 'STRING_v12'
})

# 4. Save to Excel
output_path = 'data/human_protein_shopping_list.xlsx'
try:
    # Note: Requires 'openpyxl' library installed
    shopping_list.to_excel(output_path, index=False)
    print(f"SUCCESS: Created shopping list with {len(shopping_list)} unique proteins.")
    print(f"File saved to: {output_path}")
except Exception as e:
    print(f"Error saving Excel: {e}")
    # Fallback to CSV if Excel fails
    shopping_list.to_csv('data/human_protein_shopping_list.csv', index=False)
    print("Saved as CSV instead.")

Error saving Excel: No module named 'openpyxl'
Saved as CSV instead.


In [33]:
import pandas as pd
import requests
import os
import time
from tqdm import tqdm

# 1. Setup paths
STRUCTURES_DIR = 'data/structures'
MAPPING_FILE = 'data/idmapping_2026_02_16.xlsx' # Adjust extension if it's .csv

os.makedirs(STRUCTURES_DIR, exist_ok=True)

# 2. Load the mapping file
# Using 'Entry' column from your screenshot
print(f"Reading mapping file: {MAPPING_FILE}...")
try:
    df_map = pd.read_excel(MAPPING_FILE)
except:
    # Fallback if it was saved as a CSV instead
    df_map = pd.read_csv('data/idmapping_2026_02_16.csv')

# Get unique UniProt IDs from the 'Entry' column
uniprot_ids = df_map['Entry'].dropna().unique().tolist()
print(f"Found {len(uniprot_ids)} unique UniProt IDs to download.")

# 3. Download from AlphaFold
success = 0
already_exists = 0
failed = 0

for uid in tqdm(uniprot_ids):
    # Standardize filename
    filename = f"{uid}.pdb"
    save_path = os.path.join(STRUCTURES_DIR, filename)
    
    # Skip if already downloaded
    if os.path.exists(save_path):
        already_exists += 1
        continue
    
    # AlphaFold DB Direct Download URL
    # Format: https://alphafold.ebi.ac.uk/files/AF-[UniProtID]-F1-model_v4.pdb
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb"
    
    try:
        response = requests.get(url, timeout=15)
        
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            success += 1
        else:
            # If v4 fails, try the general API (backup)
            api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uid}"
            api_res = requests.get(api_url, timeout=10)
            if api_res.status_code == 200:
                data = api_res.json()
                if data and len(data) > 0:
                    pdb_url = data[0].get('pdbUrl')
                    pdb_res = requests.get(pdb_url)
                    with open(save_path, 'wb') as f:
                        f.write(pdb_res.content)
                    success += 1
                else:
                    failed += 1
            else:
                failed += 1
        
        # Polite delay to prevent getting blocked
        time.sleep(0.05)
        
    except Exception as e:
        print(f"Error downloading {uid}: {e}")
        failed += 1

print(f"\n--- DOWNLOAD COMPLETE ---")
print(f"Successfully downloaded: {success}")
print(f"Already in folder:      {already_exists}")
print(f"Failed/Not Found:       {failed}")
print(f"Total files now in {STRUCTURES_DIR}: {len(os.listdir(STRUCTURES_DIR))}")

Reading mapping file: data/idmapping_2026_02_16.xlsx...


c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Found 721 unique UniProt IDs to download.


100%|██████████| 721/721 [26:46<00:00,  2.23s/it]


--- DOWNLOAD COMPLETE ---
Successfully downloaded: 710
Already in folder:      2
Failed/Not Found:       9
Total files now in data/structures: 717


In [1]:
%pip install fair-esm


  Using cached fair_esm-2.0.0-py3-none-any.whl.metadata (37 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser
from tqdm import tqdm
import esm
from torch_geometric.data import Data

# --- 1. CONFIGURATION ---
PDB_DIR = r'C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\structures'
OUTPUT_DIR = r'C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\processed'
DISTANCE_THRESHOLD = 10.0  # Angstroms for graph edges
PLDDT_THRESHOLD = 50.0      # Only keep high-confidence residues

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. LOAD ESM-2 MODEL ---
print("Loading ESM-2 (35M parameters) to GPU...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()

def get_protein_graph(pdb_path, protein_id):
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure(protein_id, pdb_path)
    except Exception as e:
        return None
    
    residues = []
    coords = []
    plddts = []
    sequence = ""

    # Parse PDB and Filter by pLDDT (AlphaFold confidence)
    for model_pdb in structure:
        for chain in model_pdb:
            for res in chain:
                if "CA" in res:
                    plddt = res["CA"].get_bfactor()
                    if plddt >= PLDDT_THRESHOLD:
                        residues.append(res)
                        coords.append(res["CA"].get_coord())
                        plddts.append(plddt)
                        res_name = res.get_resname()
                        # Map 3-letter to 1-letter code
                        sequence += esm.constants.proteinseq_toks.get(res_name, 'X')

    if len(sequence) < 5:
        return None

    # Generate ESM-2 Embeddings
    data_list = [(protein_id, sequence)]
    _, _, batch_tokens = batch_converter(data_list)
    batch_tokens = batch_tokens.to(device)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[12])
        # Extract residue representations (skipping start/stop tokens)
        token_representations = results["representations"][12][0, 1 : len(sequence) + 1].cpu()

    # Build Graph Edges based on 3D distance
    coords = np.array(coords)
    dist_matrix = np.linalg.norm(coords[:, np.newaxis] - coords[np.newaxis, :], axis=2)
    edge_index = np.argwhere(dist_matrix < DISTANCE_THRESHOLD)
    
    # Remove self-loops (residue connected to itself)
    edge_index = edge_index[edge_index[:, 0] != edge_index[:, 1]]
    edge_index = torch.tensor(edge_index.T, dtype=torch.long)

    return Data(
        x=token_representations,          # ESM-2 Features
        edge_index=edge_index,            # Graph Edges
        pos=torch.tensor(coords, dtype=torch.float), # 3D Coordinates
        plddt=torch.tensor(plddts, dtype=torch.float),
        protein_id=protein_id
    )

# --- 3. MAIN LOOP ---
pdb_files = [f for f in os.listdir(PDB_DIR) if f.endswith('.pdb')]
print(f"Starting processing of {len(pdb_files)} proteins...")

success_count = 0
for filename in tqdm(pdb_files):
    protein_id = filename.replace('.pdb', '')
    save_path = os.path.join(OUTPUT_DIR, f"{protein_id}.pt")
    
    if os.path.exists(save_path):
        success_count += 1
        continue
        
    try:
        pdb_path = os.path.join(PDB_DIR, filename)
        graph = get_protein_graph(pdb_path, protein_id)
        
        if graph:
            torch.save(graph, save_path)
            success_count += 1
    except Exception as e:
        print(f"Error in {protein_id}: {e}")

print(f"\n--- FEATURIZATION COMPLETE ---")
print(f"Successfully processed {success_count} proteins.")
print(f"Processed files are in: {OUTPUT_DIR}")

c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading ESM-2 (35M parameters) to GPU...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to C:\Users\Vinindu Siriwardhana/.cache\torch\hub\checkpoints\esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to C:\Users\Vinindu Siriwardhana/.cache\torch\hub\checkpoints\esm2_t12_35M_UR50D-contact-regression.pt
Starting processing of 717 proteins...


100%|██████████| 717/717 [01:41<00:00,  7.06it/s]


--- FEATURIZATION COMPLETE ---
Successfully processed 712 proteins.
Processed files are in: C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\processed


In [4]:
import os
import torch
import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1
from tqdm import tqdm
import esm
from torch_geometric.data import Data

# --- 1. CONFIGURATION ---
PDB_DIR = r'C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\structures'
OUTPUT_DIR = r'C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\processed'
DISTANCE_THRESHOLD = 10.0  
PLDDT_THRESHOLD = 50.0      
MAX_RESIDUES = 1022  

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. LOAD ESM-2 MODEL ---
print("Loading ESM-2 (35M parameters) to GPU...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_data, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model_data = model_data.to(device)
model_data.eval()
batch_converter = alphabet.get_batch_converter()

def manual_radius_graph(pos, r):
    """Pure PyTorch replacement for radius_graph that doesn't need torch-cluster"""
    # pos: [N, 3]
    # Compute pairwise distances
    dist_matrix = torch.cdist(pos, pos)
    # Find indices where distance < r
    adj = dist_matrix < r
    # Remove self-loops (diagonal)
    adj.fill_diagonal_(False)
    # Convert to edge_index format [2, E]
    return adj.nonzero().t().contiguous()

def get_protein_graph(pdb_path, protein_id):
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure(protein_id, pdb_path)
    except:
        return None
    
    coords = []
    plddts = []
    sequence = ""

    for model_pdb in structure:
        for chain in model_pdb:
            for res in chain:
                if "CA" in res:
                    plddt = res["CA"].get_bfactor()
                    res_name = res.get_resname().capitalize()
                    
                    if plddt >= PLDDT_THRESHOLD and is_aa(res_name.upper()):
                        if res_name in protein_letters_3to1:
                            aa = protein_letters_3to1[res_name]
                            sequence += aa
                            coords.append(res["CA"].get_coord())
                            plddts.append(plddt)

    if len(sequence) < 10:
        return None

    # Length Guard
    if len(sequence) > MAX_RESIDUES:
        sequence = sequence[:MAX_RESIDUES]
        coords = coords[:MAX_RESIDUES]
        plddts = plddts[:MAX_RESIDUES]

    # Generate ESM-2 Embeddings
    data_list = [("protein", sequence)]
    _, _, batch_tokens = batch_converter(data_list)
    batch_tokens = batch_tokens.to(device)

    with torch.inference_mode():
        results = model_data(batch_tokens, repr_layers=[12])
        token_reps = results["representations"][12][0, 1 : len(sequence) + 1].cpu()

    # Manual Radius Graph (No torch-cluster needed!)
    pos = torch.tensor(np.array(coords), dtype=torch.float)
    edge_index = manual_radius_graph(pos, DISTANCE_THRESHOLD)

    # Add Edge Attributes
    row, col = edge_index
    edge_attr = torch.norm(pos[row] - pos[col], dim=1).unsqueeze(1)

    torch.cuda.empty_cache()

    return Data(
        x=token_reps, 
        edge_index=edge_index,
        edge_attr=edge_attr,
        pos=pos,
        plddt=torch.tensor(plddts, dtype=torch.float),
        protein_id=protein_id,
        num_nodes=len(sequence)
    )

# --- 3. MAIN LOOP ---
pdb_files = [f for f in os.listdir(PDB_DIR) if f.endswith('.pdb')]
print(f"Starting final processing of {len(pdb_files)} proteins...")

success_count = 0
for filename in tqdm(pdb_files):
    protein_id = filename.replace('.pdb', '')
    save_path = os.path.join(OUTPUT_DIR, f"{protein_id}.pt")
    
    try:
        pdb_path = os.path.join(PDB_DIR, filename)
        graph = get_protein_graph(pdb_path, protein_id)
        
        if graph:
            torch.save(graph, save_path)
            success_count += 1
    except Exception as e:
        print(f"Error in {protein_id}: {e}")

print(f"\n--- FEATURIZATION COMPLETE ---")
print(f"Successfully processed {success_count} proteins.")

Loading ESM-2 (35M parameters) to GPU...
Starting final processing of 717 proteins...


100%|██████████| 717/717 [01:14<00:00,  9.61it/s]


--- FEATURIZATION COMPLETE ---
Successfully processed 712 proteins.


In [10]:
import os

folder_path = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data"
print(os.listdir(folder_path))


['dengue_structures', 'idmapping_2026_02_16.xlsx', 'interaction_master_list.csv', 'Mapping Unnecessary', 'pdbbind', 'processed', 'splits', 'structures']


In [12]:
import pandas as pd

folder_path = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data"
mapping_file = f"{folder_path}\\idmapping_2026_02_16.xlsx"

mapping_df = pd.read_excel(mapping_file)

# See the exact column names
print(mapping_df.columns.tolist())


['From', 'Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names', 'Organism', 'Length', '3D']


c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [13]:
import pandas as pd

# --- 1. Folder path ---
folder_path = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data"

# --- 2. Load mapping file ---
mapping_file = f"{folder_path}\\idmapping_2026_02_16.xlsx"
mapping_df = pd.read_excel(mapping_file)

# Strip any leading/trailing spaces from column names
mapping_df.columns = mapping_df.columns.str.strip()

# Create dictionary: ENSP -> UniProt
mapping_dict = dict(zip(mapping_df['From'], mapping_df['Entry']))

# --- 3. Load PPI file (CSV) ---
ppi_file = f"{folder_path}\\interaction_master_list.csv"
ppi_df = pd.read_csv(ppi_file)

# Strip spaces from PPI columns just in case
ppi_df.columns = ppi_df.columns.str.strip()

# --- 4. Map ENSP IDs to UniProt IDs ---
ppi_df['p1_UniProt'] = ppi_df['p1'].map(mapping_dict)
ppi_df['p2_UniProt'] = ppi_df['p2'].map(mapping_dict)

# --- 5. Drop interactions where mapping failed (optional) ---
ppi_df = ppi_df.dropna(subset=['p1_UniProt', 'p2_UniProt'])

# --- 6. Save the new file as CSV ---
output_file = f"{folder_path}\\interaction_master_list_uniprot.csv"
ppi_df.to_csv(output_file, index=False)

print("✅ Done! New file saved as 'interaction_master_list_uniprot.csv'")


c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ Done! New file saved as 'interaction_master_list_uniprot.csv'


In [16]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader  # Changed to standard PyTorch loader
from torch_geometric.data import Dataset, Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# --- 1. DATASET & COLLATOR ---
class DenguePPIDataset(Dataset):
    def __init__(self, dataframe, root_dir):
        super(DenguePPIDataset, self).__init__(root_dir)
        self.interactions = dataframe.reset_index(drop=True)
        self.root_dir = root_dir

    def len(self):
        return len(self.interactions)

    def get(self, idx):
        row = self.interactions.iloc[idx]
        # weights_only=False handles the PyTorch 2.6+ security update
        data1 = torch.load(os.path.join(self.root_dir, f"{row['p1_UniProt']}.pt"), weights_only=False)
        data2 = torch.load(os.path.join(self.root_dir, f"{row['p2_UniProt']}.pt"), weights_only=False)
        
        # Explicitly cast label to int to avoid numpy.int64 errors
        label = int(row['label'])
        return data1, data2, label

def pair_collate(samples):
    """Custom collate to handle pairs of graphs for Siamese networks."""
    data1_list, data2_list, labels = zip(*samples)
    # Batch.from_data_list merges individual graphs into one large batch graph
    batch1 = Batch.from_data_list(data1_list)
    batch2 = Batch.from_data_list(data2_list)
    labels = torch.tensor(labels, dtype=torch.float).view(-1, 1)
    return batch1, batch2, labels

# --- 2. DATA PREPARATION ---
def prepare_data(csv_path, processed_dir):
    df = pd.read_csv(csv_path)
    
    # Filter only rows where both proteins exist in the processed folder
    processed_files = {f.replace('.pt', '') for f in os.listdir(processed_dir) if f.endswith('.pt')}
    df = df[df['p1_UniProt'].isin(processed_files) & df['p2_UniProt'].isin(processed_files)]
    
    # Split by unique Human Protein ID (p2_UniProt) to prevent data leakage
    human_prots = df['p2_UniProt'].unique()
    train_prots, temp_prots = train_test_split(human_prots, test_size=0.30, random_state=42)
    val_prots, test_prots = train_test_split(temp_prots, test_size=0.50, random_state=42)
    
    train_df = df[df['p2_UniProt'].isin(train_prots)]
    val_df = df[df['p2_UniProt'].isin(val_prots)]
    test_df = df[df['p2_UniProt'].isin(test_prots)]
    
    # Calculate pos_weight for BCE Loss
    pos_count = train_df['label'].sum()
    neg_count = len(train_df) - pos_count
    pos_weight = torch.tensor([neg_count / pos_count])
    
    print(f"Dataset Split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
    return train_df, val_df, test_df, pos_weight

# --- 3. ARCHITECTURE ---
class SiameseGATv2(nn.Module):
    def __init__(self, node_features=480, hidden_dim=256, heads=4):
        super(SiameseGATv2, self).__init__()
        self.conv1 = GATv2Conv(node_features, hidden_dim, heads=heads, edge_dim=1)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=1)
        
        pooling_dim = hidden_dim * 2 
        self.fc = nn.Sequential(
            nn.Linear(pooling_dim * 4, 512), 
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def encode(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = torch.relu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = torch.relu(self.conv2(x, edge_index, edge_attr=edge_attr))
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def forward(self, data1, data2):
        emb1, emb2 = self.encode(data1), self.encode(data2)
        combined = torch.cat([emb1, emb2, torch.abs(emb1 - emb2), emb1 * emb2], dim=1)
        return self.fc(combined)

# --- 4. EVALUATION LOGIC ---
def run_evaluation(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for b1, b2, labels in loader:
            logits = model(b1.to(device), b2.to(device))
            all_preds.append(torch.sigmoid(logits).cpu())
            all_labels.append(labels)
    
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return roc_auc_score(labels, preds), average_precision_score(labels, preds)

# --- 5. EXECUTION ---
if __name__ == "__main__":
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    BATCH_SIZE = 16
    MAX_EPOCHS = 100
    PATIENCE = 10
    
    # Paths (Windows Raw Strings)
    PROCESSED_DIR = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\processed"
    CSV_FILE = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\interaction_master_list_uniprot.csv"

    # Prep Data
    train_df, val_df, test_df, pos_weight = prepare_data(CSV_FILE, PROCESSED_DIR)
    
    # Loader settings (Using standard torch.utils.data.DataLoader)
    loader_args = {
        'batch_size': BATCH_SIZE, 
        'num_workers': 0, 
        'pin_memory': True, 
        'collate_fn': pair_collate
    }
    
    train_loader = DataLoader(DenguePPIDataset(train_df, PROCESSED_DIR), shuffle=True, **loader_args)
    val_loader = DataLoader(DenguePPIDataset(val_df, PROCESSED_DIR), shuffle=False, **loader_args)
    test_loader = DataLoader(DenguePPIDataset(test_df, PROCESSED_DIR), shuffle=False, **loader_args)

    # Initialize
    model = SiameseGATv2().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

    best_val_auc = 0
    epochs_no_improve = 0

    print(f"Training started on {DEVICE}...")
    
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for b1, b2, labels in train_loader:
            optimizer.zero_grad()
            logits = model(b1.to(DEVICE), b2.to(DEVICE))
            loss = criterion(logits, labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        val_auc, val_ap = run_evaluation(model, val_loader, DEVICE)
        
        print(f"Epoch {epoch:03d} | Loss: {epoch_loss/len(train_loader):.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_dengue_gnn.pth')
            print(">>> New Best Model Saved")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    # Final Test
    print("\n--- Final Test ---")
    if os.path.exists('best_dengue_gnn.pth'):
        model.load_state_dict(torch.load('best_dengue_gnn.pth'))
        test_auc, test_ap = run_evaluation(model, test_loader, DEVICE)
        print(f"FINAL TEST RESULTS | AUC: {test_auc:.4f} | Avg Precision: {test_ap:.4f}")

Dataset Split: Train=39722, Val=8230, Test=8230
Training started on cuda...


c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 001 | Loss: 0.4996 | Val AUC: 0.7918 | Val AP: 0.7678
>>> New Best Model Saved


c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 002 | Loss: 0.4071 | Val AUC: 0.8291 | Val AP: 0.8001
>>> New Best Model Saved


...

Epoch 019 | Loss: 0.1825 | Val AUC: 0.8808 | Val AP: 0.8448


c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 020 | Loss: 0.1759 | Val AUC: 0.8763 | Val AP: 0.8348


c:\Users\Vinindu Siriwardhana\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch_geometric.data import Dataset, Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

# --- 1. ARCHITECTURE ---
class SiameseGATv2(nn.Module):
    def __init__(self, node_features=480, hidden_dim=256, heads=4):
        super(SiameseGATv2, self).__init__()
        self.conv1 = GATv2Conv(node_features, hidden_dim, heads=heads, edge_dim=1)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=1)
        
        pooling_dim = hidden_dim * 2 
        self.fc = nn.Sequential(
            nn.Linear(pooling_dim * 4, 512), 
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def encode(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = torch.relu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = torch.relu(self.conv2(x, edge_index, edge_attr=edge_attr))
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def forward(self, data1, data2):
        emb1, emb2 = self.encode(data1), self.encode(data2)
        combined = torch.cat([emb1, emb2, torch.abs(emb1 - emb2), emb1 * emb2], dim=1)
        return self.fc(combined)

# --- 2. DATA UTILITIES ---
class DenguePPIDataset(Dataset):
    def __init__(self, dataframe, root_dir):
        super(DenguePPIDataset, self).__init__(root_dir)
        self.interactions = dataframe.reset_index(drop=True)
        self.root_dir = root_dir

    def len(self):
        return len(self.interactions)

    def get(self, idx):
        row = self.interactions.iloc[idx]
        data1 = torch.load(os.path.join(self.root_dir, f"{row['p1_UniProt']}.pt"), weights_only=False)
        data2 = torch.load(os.path.join(self.root_dir, f"{row['p2_UniProt']}.pt"), weights_only=False)
        label = int(row['label'])
        return data1, data2, label

def pair_collate(samples):
    data1_list, data2_list, labels = zip(*samples)
    batch1 = Batch.from_data_list(data1_list)
    batch2 = Batch.from_data_list(data2_list)
    labels = torch.tensor(labels, dtype=torch.float).view(-1, 1)
    return batch1, batch2, labels

# --- 3. CONFIG ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROCESSED_DIR = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\processed"
CSV_FILE      = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\data\interaction_master_list_uniprot.csv"
MODEL_PATH    = r"C:\Users\Vinindu Siriwardhana\Documents\DengueGNN\best_dengue_gnn.pth"

# --- 4. LOAD & SPLIT DATA ---
df = pd.read_csv(CSV_FILE)
processed_files = {f.replace('.pt', '') for f in os.listdir(PROCESSED_DIR) if f.endswith('.pt')}
df = df[df['p1_UniProt'].isin(processed_files) & df['p2_UniProt'].isin(processed_files)]

human_prots = df['p2_UniProt'].unique()
train_prots, temp_prots = train_test_split(human_prots, test_size=0.30, random_state=42)
val_prots, test_prots   = train_test_split(temp_prots,  test_size=0.50, random_state=42)
test_df = df[df['p2_UniProt'].isin(test_prots)]

test_loader = DataLoader(
    DenguePPIDataset(test_df, PROCESSED_DIR),
    batch_size=16,
    shuffle=False,
    collate_fn=pair_collate
)

# --- 5. LOAD MODEL ---
model = SiameseGATv2().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f"Model loaded successfully | Evaluating on {DEVICE}")

# --- 6. INFERENCE ---
all_preds  = []
all_labels = []

print(f"Evaluating {len(test_df)} pairs...")
with torch.no_grad():
    for b1, b2, labels in test_loader:
        logits = model(b1.to(DEVICE), b2.to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.numpy())

# --- 7. METRICS ---
auc          = roc_auc_score(all_labels, all_preds)
ap           = average_precision_score(all_labels, all_preds)
binary_preds = [1 if p >= 0.5 else 0 for p in all_preds]

print("\n" + "="*40)
print("    DENGUE GNN FINAL TEST REPORT")
print("="*40)
print(f"Test AUC:               {auc:.4f}")
print(f"Test Average Precision: {ap:.4f}")
print("-"*40)
print("Classification Report:")
print(classification_report(all_labels, binary_preds, target_names=['No Interaction', 'Interaction']))
print("-"*40)
print("Confusion Matrix:")
print(confusion_matrix(all_labels, binary_preds))
print("="*40)

Model loaded successfully | Evaluating on cuda
Evaluating 8230 pairs...

    DENGUE GNN FINAL TEST REPORT
Test AUC:               0.8729
Test Average Precision: 0.8348
----------------------------------------
Classification Report:
                precision    recall  f1-score   support

No Interaction       0.75      0.85      0.80      4183
   Interaction       0.82      0.71      0.76      4047

      accuracy                           0.78      8230
     macro avg       0.79      0.78      0.78      8230
  weighted avg       0.78      0.78      0.78      8230

----------------------------------------
Confusion Matrix:
[[3557  626]
 [1184 2863]]


: 

In [3]:
# Debug NS1 processing
from Bio.PDB import PDBParser
from Bio.Data.PDBData import protein_letters_3to1

parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", r"data/raw/ns1_dengue.pdb")

coords, sequence = [], ""
for model_struct in structure:
    for chain in model_struct:
        for res in chain:
            if "CA" in res:
                bfactor = res["CA"].get_bfactor()
                resname = res.get_resname()
                aa = protein_letters_3to1.get(resname.upper(), 'X')
                coords.append(res["CA"].get_coord())
                sequence += aa

print(f"Sequence length: {len(sequence)}")
print(f"Coord count: {len(coords)}")
print(f"First 20 residues: {sequence[:20]}")

Sequence length: 642
Coord count: 642
First 20 residues: ADSGCVVSKELKCGSGIFIT


In [4]:
# Check what bfactor values NS1 actually has
bfactors = [res["CA"].get_bfactor() for model in structure for chain in model for res in chain if "CA" in res]
print(f"Min: {min(bfactors):.1f}, Max: {max(bfactors):.1f}, Mean: {sum(bfactors)/len(bfactors):.1f}")

Min: 46.2, Max: 158.5, Mean: 87.9


In [5]:
# Count how many residues actually pass the filter
passed = [res for model in structure for chain in model for res in chain 
          if "CA" in res and res["CA"].get_bfactor() > 50]
print(f"Residues passing filter: {len(passed)} out of {len(coords)}")

Residues passing filter: 638 out of 642


In [ ]:
import os
import glob
import gzip
import torch
import csv
import numpy as np
import pandas as pd
from tqdm import tqdm
from Bio.PDB import PDBParser
from Bio.Data.PDBData import protein_letters_3to1
import torch.nn as nn
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool, radius_graph
from torch_geometric.data import Data
import esm

# --- 1. ARCHITECTURE ---
class SiameseGATv2(nn.Module):
    def __init__(self, node_features=480, hidden_dim=256, heads=4):
        super(SiameseGATv2, self).__init__()
        self.conv1 = GATv2Conv(node_features, hidden_dim, heads=heads, edge_dim=1)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=1)
        pooling_dim = hidden_dim * 2
        self.fc = nn.Sequential(
            nn.Linear(pooling_dim * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def encode(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = torch.relu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = torch.relu(self.conv2(x, edge_index, edge_attr=edge_attr))
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def forward(self, data1, data2):
        emb1, emb2 = self.encode(data1), self.encode(data2)
        combined = torch.cat([emb1, emb2, torch.abs(emb1 - emb2), emb1 * emb2], dim=1)
        return self.fc(combined)

# --- 2. GRAPH PREPROCESSOR ---
def preprocess_pdb_to_graph(pdb_path, esm_model, converter, esm_device, geo_device, filter_plddt=True):
    parser = PDBParser(QUIET=True)
    try:
        if pdb_path.endswith(".gz"):
            with gzip.open(pdb_path, "rt") as f:
                structure = parser.get_structure("protein", f)
        else:
            structure = parser.get_structure("protein", pdb_path)

        coords, sequence = [], ""
        for model_struct in structure:
            for chain in model_struct:
                for res in chain:
                    if "CA" in res:
                        if filter_plddt and res["CA"].get_bfactor() <= 50:
                            continue
                        coords.append(res["CA"].get_coord())
                        sequence += protein_letters_3to1.get(res.get_resname().upper(), 'X')

        if len(sequence) < 5 or len(sequence) > 1022:
            print(f"  Skipped {os.path.basename(pdb_path)}: length {len(sequence)}")
            return None

        # ESM-2 on CPU
        _, _, batch_tokens = converter([("protein", sequence)])
        with torch.no_grad():
            results = esm_model(batch_tokens.to(esm_device), repr_layers=[12])
            x = results["representations"][12][0, 1 : len(sequence) + 1].cpu()

        # Geometry on GPU
        pos = torch.tensor(np.array(coords), dtype=torch.float).to(geo_device)
        edge_index = radius_graph(pos, r=10.0, loop=False)
        dist = torch.norm(pos[edge_index[0]] - pos[edge_index[1]], dim=-1, keepdim=True)
        edge_attr = 1.0 / (dist + 1e-6)

        return Data(x=x, edge_index=edge_index.cpu(), edge_attr=edge_attr.cpu())

    except Exception as e:
        print(f"  ERROR in {os.path.basename(pdb_path)}: {e}")
        return None

# --- 3. EXECUTION ---
GPU_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ESM_DEVICE = torch.device('cpu')
print(f"GNN running on: {GPU_DEVICE} | ESM running on: {ESM_DEVICE}")

# Load ESM on CPU
esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
esm_model = esm_model.to(ESM_DEVICE).eval()
batch_converter = alphabet.get_batch_converter()

# Load GNN on GPU
model = SiameseGATv2().to(GPU_DEVICE)
model.load_state_dict(torch.load("best_dengue_gnn.pth", map_location=GPU_DEVICE, weights_only=False))
model.eval()
print("Models loaded successfully.")

# NS1 query — filter_plddt=False because it's an experimental PDB, not AlphaFold
ns1_path = r"data/raw/ns1_dengue.pdb"
print("Processing Dengue NS1 structure...")
ns1_graph = preprocess_pdb_to_graph(ns1_path, esm_model, batch_converter, ESM_DEVICE, GPU_DEVICE, filter_plddt=False)

if ns1_graph is None:
    raise RuntimeError("NS1 graph failed to process. Check the PDB file and path.")

ns1_graph = ns1_graph.to(GPU_DEVICE)
ns1_graph.batch = torch.zeros(ns1_graph.x.shape[0], dtype=torch.long).to(GPU_DEVICE)
print(f"NS1 graph ready: {ns1_graph.x.shape[0]} residues, {ns1_graph.edge_index.shape[1]} edges.")

# Crash recovery
output_csv = "ns1_human_interactome_results.csv"
if os.path.exists(output_csv):
    done = set(pd.read_csv(output_csv)['uniprot_id'].astype(str))
    print(f"Resuming — {len(done)} proteins already processed.")
else:
    done = set()
    with open(output_csv, "w", newline="") as f:
        csv.writer(f).writerow(["uniprot_id", "interaction_score"])

# Proteome screening
human_files = glob.glob(r"data/raw/human_proteome/*.pdb.gz")
print(f"Screening {len(human_files)} human proteins against NS1...")

skipped = 0
for pdb_file in tqdm(human_files):
    uniprot_id = os.path.basename(pdb_file).split("-")[1]

    if uniprot_id in done:
        continue

    h_graph = preprocess_pdb_to_graph(pdb_file, esm_model, batch_converter, ESM_DEVICE, GPU_DEVICE)

    if h_graph is not None:
        h_graph = h_graph.to(GPU_DEVICE)
        h_graph.batch = torch.zeros(h_graph.x.shape[0], dtype=torch.long).to(GPU_DEVICE)

        with torch.no_grad():
            score = torch.sigmoid(model(ns1_graph, h_graph)).item()

        with open(output_csv, "a", newline="") as f:
            csv.writer(f).writerow([uniprot_id, score])

        del h_graph
        torch.cuda.empty_cache()
    else:
        skipped += 1

# Sort and save
df = pd.read_csv(output_csv).sort_values(by="interaction_score", ascending=False)
df.to_csv(output_csv, index=False)
print(f"\nDiscovery Phase Complete.")
print(f"Screened: {len(df)} proteins | Skipped: {skipped}")
print(f"Top hit: {df.iloc[0]['uniprot_id']} with score {df.iloc[0]['interaction_score']:.4f}")
print(f"Results saved to: {output_csv}")

In [ ]:
import os
import glob
import gzip
import torch
import csv
import numpy as np
import pandas as pd
from tqdm import tqdm
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1
import torch.nn as nn
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Data
import esm

# --- 1. ARCHITECTURE ---
class SiameseGATv2(nn.Module):
    def __init__(self, node_features=480, hidden_dim=256, heads=4):
        super(SiameseGATv2, self).__init__()
        self.conv1 = GATv2Conv(node_features, hidden_dim, heads=heads, edge_dim=1)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=1)
        pooling_dim = hidden_dim * 2
        self.fc = nn.Sequential(
            nn.Linear(pooling_dim * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def encode(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = torch.relu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = torch.relu(self.conv2(x, edge_index, edge_attr=edge_attr))
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def forward(self, data1, data2):
        emb1, emb2 = self.encode(data1), self.encode(data2)
        combined = torch.cat([emb1, emb2, torch.abs(emb1 - emb2), emb1 * emb2], dim=1)
        return self.fc(combined)

# --- 2. PURE PYTORCH RADIUS GRAPH ---
def build_radius_graph(coords, radius=10.0):
    pos = torch.tensor(np.array(coords), dtype=torch.float)
    dist_matrix = torch.cdist(pos, pos)
    adj = dist_matrix < radius
    adj.fill_diagonal_(False)
    edge_index = adj.nonzero().t().contiguous()
    # ✅ Raw distance — matches training code exactly
    row, col = edge_index
    edge_attr = torch.norm(pos[row] - pos[col], dim=1).unsqueeze(1)
    return edge_index, edge_attr

# --- 3. GRAPH PREPROCESSOR ---
def preprocess_pdb_to_graph(pdb_path, esm_model, converter, esm_device, filter_plddt=True):
    parser = PDBParser(QUIET=True)
    try:
        if pdb_path.endswith(".gz"):
            with gzip.open(pdb_path, "rt") as f:
                structure = parser.get_structure("protein", f)
        else:
            structure = parser.get_structure("protein", pdb_path)

        coords, sequence = [], ""
        for model_struct in structure:
            for chain in model_struct:
                for res in chain:
                    if "CA" in res:
                        if filter_plddt and res["CA"].get_bfactor() < 50:
                            continue
                        # ✅ .capitalize() and is_aa check — matches training code exactly
                        res_name = res.get_resname().capitalize()
                        if is_aa(res_name.upper()) and res_name in protein_letters_3to1:
                            sequence += protein_letters_3to1[res_name]
                            coords.append(res["CA"].get_coord())

        if len(sequence) < 10 or len(sequence) > 1022:
            print(f"  Skipped {os.path.basename(pdb_path)}: length {len(sequence)}")
            return None

        # ESM-2 on CPU
        _, _, batch_tokens = converter([("protein", sequence)])
        with torch.no_grad():
            results = esm_model(batch_tokens.to(esm_device), repr_layers=[12])
            x = results["representations"][12][0, 1 : len(sequence) + 1].cpu()

        # Geometry — pure PyTorch, no torch-cluster
        edge_index, edge_attr = build_radius_graph(coords)

        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    except Exception as e:
        print(f"  ERROR in {os.path.basename(pdb_path)}: {e}")
        return None

# --- 4. EXECUTION ---
GPU_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ESM_DEVICE = torch.device('cpu')
print(f"GNN running on: {GPU_DEVICE} | ESM running on: {ESM_DEVICE}")

# Load ESM on CPU
esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
esm_model = esm_model.to(ESM_DEVICE).eval()
batch_converter = alphabet.get_batch_converter()

# Load GNN on GPU
model = SiameseGATv2().to(GPU_DEVICE)
model.load_state_dict(torch.load("best_dengue_gnn.pth", map_location=GPU_DEVICE, weights_only=False))
model.eval()
print("Models loaded successfully.")

# NS1 query — filter_plddt=False (experimental PDB, not AlphaFold)
ns1_path = r"data/raw/ns1_dengue.pdb"
print("Processing Dengue NS1 structure...")
ns1_graph = preprocess_pdb_to_graph(ns1_path, esm_model, batch_converter, ESM_DEVICE, filter_plddt=False)

if ns1_graph is None:
    raise RuntimeError("NS1 graph failed to process. Check the error message above.")

ns1_graph = ns1_graph.to(GPU_DEVICE)
ns1_graph.batch = torch.zeros(ns1_graph.x.shape[0], dtype=torch.long).to(GPU_DEVICE)
print(f"NS1 ready: {ns1_graph.x.shape[0]} residues, {ns1_graph.edge_index.shape[1]} edges.")

# Crash recovery — resume if output already exists
output_csv = "ns1_human_interactome_results.csv"
if os.path.exists(output_csv):
    done = set(pd.read_csv(output_csv)['uniprot_id'].astype(str))
    print(f"Resuming — {len(done)} proteins already processed.")
else:
    done = set()
    with open(output_csv, "w", newline="") as f:
        csv.writer(f).writerow(["uniprot_id", "interaction_score"])

# Proteome screening
human_files = glob.glob(r"data/raw/human_proteome/*.pdb.gz")
print(f"Screening {len(human_files)} human proteins against NS1...")

skipped = 0
for pdb_file in tqdm(human_files):
    uniprot_id = os.path.basename(pdb_file).split("-")[1]

    if uniprot_id in done:
        continue

    h_graph = preprocess_pdb_to_graph(pdb_file, esm_model, batch_converter, ESM_DEVICE)

    if h_graph is not None:
        h_graph = h_graph.to(GPU_DEVICE)
        h_graph.batch = torch.zeros(h_graph.x.shape[0], dtype=torch.long).to(GPU_DEVICE)

        with torch.no_grad():
            score = torch.sigmoid(model(ns1_graph, h_graph)).item()

        with open(output_csv, "a", newline="") as f:
            csv.writer(f).writerow([uniprot_id, score])

        del h_graph
        torch.cuda.empty_cache()
    else:
        skipped += 1

# Sort and save
df = pd.read_csv(output_csv).sort_values(by="interaction_score", ascending=False)
df.to_csv(output_csv, index=False)
print(f"\nDiscovery Phase Complete.")
print(f"Screened: {len(df)} proteins | Skipped: {skipped}")
print(f"Top hit: {df.iloc[0]['uniprot_id']} with score {df.iloc[0]['interaction_score']:.4f}")
print(f"Results saved to: {output_csv}")

In [10]:
import pandas as pd

df = pd.read_csv("ns1_human_interactome_results.csv")
print(f"Total proteins screened: {len(df)}")
print(f"\nTop 20 predicted NS1 interactors:\n")
print(df.head(20).to_string(index=False))

Total proteins screened: 20565

Top 20 predicted NS1 interactors:

uniprot_id  interaction_score
    P31749           0.952697
    P98088           0.942741
    P01375           0.941579
    P35222           0.934173
    P98088           0.934145
    P98088           0.930076
    P98088           0.921300
    P0CG47           0.918806
    P0CG48           0.917486
    Q9HC84           0.913719
    P98088           0.911678
    P63261           0.905711
    P60709           0.904707
    Q8TC94           0.903742
    P62736           0.900065
    P63267           0.898295
    Q15848           0.898161
    P01106           0.897558
    P68032           0.897477
    P68133           0.897336


In [11]:
# Group by ID and take the highest score for each protein
df_unique = df.groupby('uniprot_id')['interaction_score'].max().sort_values(ascending=False).reset_index()
print(df_unique.head(20))

   uniprot_id  interaction_score
0      P31749           0.952697
1      P98088           0.942741
2      P01375           0.941579
3      P35222           0.934173
4      P0CG47           0.918806
5      P0CG48           0.917486
6      Q9HC84           0.913719
7      P63261           0.905711
8      P60709           0.904707
9      Q8TC94           0.903742
10     P62736           0.900065
11     P63267           0.898295
12     Q15848           0.898161
13     P01106           0.897558
14     P68032           0.897477
15     P68133           0.897336
16     Q16665           0.895958
17     Q562R1           0.893762
18     P03372           0.891100
19     Q9BYX7           0.888493


In [12]:
# Group by ID and take the highest score for each protein
df_unique = df.groupby('uniprot_id')['interaction_score'].max().sort_values(ascending=False).reset_index()
print(df_unique.head(200))

    uniprot_id  interaction_score
0       P31749           0.952697
1       P98088           0.942741
2       P01375           0.941579
3       P35222           0.934173
4       P0CG47           0.918806
..         ...                ...
195     P48146           0.589501
196     P08670           0.588623
197     P01112           0.586843
198     P30740           0.586423
199     P10720           0.585235

[200 rows x 2 columns]


In [13]:
import pandas as pd
import requests

# 1. Load the discovery results
input_csv = "ns1_human_interactome_results.csv"
output_csv = "top_200_ns1_interactors.csv"

df = pd.read_csv(input_csv)

# 2. Group by UniProt ID and keep only the highest score for each protein
# This removes duplicates caused by AlphaFold fragments
df_unique = df.groupby('uniprot_id', as_index=False)['interaction_score'].max()

# 3. Sort by score descending and take top 200
top_200 = df_unique.sort_values(by='interaction_score', ascending=False).head(200)

print(f"Filtered {len(df)} entries down to {len(top_200)} unique top proteins.")

# 4. (Optional but Recommended) Fetch Gene Names for easier interpretation
print("Fetching Gene Names from UniProt...")

def get_gene_name(uniprot_id):
    try:
        url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            gene_name = data['genes'][0]['geneName']['value']
            description = data['proteinDescription']['recommendedName']['fullName']['value']
            return gene_name, description
    except:
        return "N/A", "N/A"
    return "N/A", "N/A"

# Apply the mapping (Note: this might take a minute for 200 IDs)
gene_info = [get_gene_name(uid) for uid in top_200['uniprot_id']]
top_200['gene_name'] = [info[0] for info in gene_info]
top_200['protein_description'] = [info[1] for info in gene_info]

# 5. Save the final discovery list
top_200.to_csv(output_csv, index=False)

print(f"\nSuccess! Top 200 unique hits saved to: {output_csv}")
print(top_200[['uniprot_id', 'gene_name', 'interaction_score']].head(20))

Filtered 20565 entries down to 200 unique top proteins.
Fetching Gene Names from UniProt...

Success! Top 200 unique hits saved to: top_200_ns1_interactors.csv
      uniprot_id gene_name  interaction_score
5282      P31749      AKT1           0.952697
6979      P98088    MUC5AC           0.942741
3248      P01375       N/A           0.941579
5390      P35222    CTNNB1           0.934173
3937      P0CG47       UBB           0.918806
3938      P0CG48       UBC           0.917486
17045     Q9HC84     MUC5B           0.913719
6809      P63261     ACTG1           0.905711
6607      P60709      ACTB           0.904707
13342     Q8TC94     ACTL9           0.903742
6743      P62736     ACTA2           0.900065
6810      P63267     ACTG2           0.898295
8308      Q15848    ADIPOQ           0.898161
3213      P01106       MYC           0.897558
6820      P68032     ACTC1           0.897477
6824      P68133     ACTA1           0.897336
8426      Q16665     HIF1A           0.895958
8974      Q5